In [1]:
import numpy as np
import scipy.special as scp
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
plt.style.use('../include/aps.mplstyle')

In [2]:
#Load the data it is in the form x,y,z,U
data = np.loadtxt('../data/He_TPI_samples_xyz.dat',skiprows=1)

In [3]:
#Ok now that we have a fitted form let us look at the noise:
def fittedpot(r):
    R = 14.4228
    sigma = 3.1466
    densityepsilon =  0.5786
    offset = 0.6018
    x = r / R;
    x2 = x*x;
    x4 = x2*x2;
    x6 = x2*x4;
    x8 = x4*x4;
    f1 = 1.0 / (1.0 - x2);
    sigoR3 = pow(sigma/R,3.0);
    sigoR9 = sigoR3*sigoR3*sigoR3;
    v9 = (1.0*pow(f1,9.0)/(240.0)) * ((1091.0 + 11156*x2 + 16434*x4 + 4052*x6 + 35*x8)*scp.ellipe(x2) - 8.0*(1.0 - x2)*(1.0 + 7*x2)*(97.0 + 134*x2 + 25*x4)*scp.ellipk(x2));
    v3 = 2.0*pow(f1,3.0) * ((7.0 + x2)*scp.ellipe(x2) - 4.0*(1.0-x2)*scp.ellipk(x2));
    val = (np.pi*densityepsilon*sigma*sigma*sigma/3.0)*(sigoR9*v9 - sigoR3*v3)
    return val + offset

In [4]:
#Now take an average along z
xy = data[:, :2]
# Unique (x,y) groups
unique_xy, inverse = np.unique(xy, axis=0, return_inverse=True)
# Average V over each group
V_avg = np.bincount(inverse, weights=data[:, 3]) / np.bincount(inverse)
# Final array: (x, y, V_avg)
data2d = np.column_stack((unique_xy, V_avg))
x0 = 42.996
y0 = 39.413
x = data2d[:, 0]
y = data2d[:, 1]
V = data2d[:, 2]
r = np.sqrt((x - x0)**2 + (y - y0)**2)
theta = np.arctan2(y-y0, x-x0)  
#Realistically we only care about radial data less than 14 then
x_fil = x[r < 14.3]
y_fil = y[r < 14.3]
r_fil = r[r < 14.3]
theta_fil = theta[r < 14.3]
V_fil = np.clip(V[r < 14.3], a_min=None, a_max=200)
theta_fil = np.arctan2(y_fil - y0,x_fil-x0)
FittedR = np.clip(fittedpot(r_fil), a_min=None, a_max=200)
deltaV = FittedR-V_fil
print(FittedR.max(),FittedR.min(),V_fil.max(),V_fil.min(),deltaV.max())
deltaV = FittedR-V_fil

200.0 -46.50646652210412 200.0 -94.18073422664264 294.18073422664264


In [5]:
dr_tol = 1e-5
    
r_sorted  = np.sort(np.unique(np.round(r_fil / dr_tol) * dr_tol))
NR = len(np.unique(r))
    
shells = {}
for r0 in r_sorted:
    mask = np.abs(r_fil - r0) < dr_tol
    if mask.sum() < 3:
        continue
    idx = np.where(mask)[0]
    # sort by theta within each shell
    order = np.argsort(theta_fil[idx])
    shells[r0] = {'theta': theta_fil[idx][order], 'V': V_fil[idx][order]}

r_keys = np.array(sorted(shells.keys()))
NR = len(r_keys)
NT = 144  # points per shell
    
# build V_2d and theta_2d by stacking shells in r order
V_2d     = np.zeros((NR, NT))
theta_2d = np.zeros((NR, NT))
    
for i, r0 in enumerate(r_keys):
    s = shells[r0]
    th = s['theta'] % (2 * np.pi)   # wrap (-π, π] → [0, 2π)
    order = np.argsort(th)           # re-sort after wrapping
    V_2d[i, :]     = s['V'][order]
    theta_2d[i, :] = th[order]
    
# use the first shell's theta as the common grid
theta = theta_2d[0, :]
    
# Fix 2: check all shells have consistent theta after wrapping
theta_0 = theta_2d[0, :]
max_theta_diff = max(np.max(np.abs(theta_2d[i] - theta_0)) for i in range(NR))
    
# now compute dV
    
dV = V_2d                         
    
# and the decomposition
theta = theta_2d[0]   # shape (NT,) — use first shell's theta as the common grid
coeffs = np.fft.rfft(V_2d, axis=1) / NT   
    
An = 2 * coeffs.real    
Bn = -2 * coeffs.imag    # shape (NR, NT//2+1)
Cn = np.abs(coeffs) * 2 # shape (NR, NT//2+1)
thetan = np.angle(coeffs)

    
mask = (r_keys >= 10) & (r_keys <= 14.4)
    

A_mean      = An[mask, 1:].mean(axis=0)       # shape (NT//2,)
B_mean      = Bn[mask, 1:].mean(axis=0)       # shape (NT//2,)

In [7]:
with plt.style.context('../include/aps.mplstyle'):
    figsize = plt.rcParams['figure.figsize']
    fig,axs = plt.subplots(3,1,figsize=(figsize[0],3*figsize[1]), constrained_layout=True)
    #fig = plt.figure(figsize=(figsize[0], 3*figsize[1]), constrained_layout=True)
    
    tp1 = axs[0].tripcolor(x_fil - x0,y_fil - y0,deltaV,vmax = -deltaV.min(), cmap='bwr_r')
    cb = fig.colorbar(tp1, ax=axs[0], cax=axs[0].inset_axes([0.9, 0.05, 0.03, 0.4]))
    cb.ax.yaxis.set_ticks_position('left')
    cb.ax.yaxis.set_label_position('right')
    cb.set_label(r'$\delta U$ [K]')
    axs[0].set_ylabel(r'$y$[\AA]')
    axs[0].set_xlabel(r'$x$[\AA]')
    axs[0].set_xlim(-17.9,19.9)
    axs[0].set_ylim(-14.99,14.99)
    axs[0].text(-17,11,'(a)')
    axs[1].bar(range(1, len(A_mean)+1), A_mean)
    axs[1].set_xlabel('Fourier mode n')
    axs[1].set_ylabel('mean $A_n$ [K]')
    axs[1].set_xlim(-2,72)
    axs[1].text(-1,14,'(b)')
    ax_inset1 = axs[1].inset_axes([0.5, 0.67, 0.45, 0.3])
    ax_inset1.plot(r_keys[mask], An[mask, 1], label=r'$A_1$')
    ax_inset1.plot(r_keys[mask], An[mask, 2], label=r'$A_{2}$')
    ax_inset1.set_xlabel(r'$r$[\AA]')
    ax_inset1.set_ylabel(r'$A_n(r)$[K]')
    ax_inset1.set_ylim(-20,65)
    ax_inset1.set_yticks([0,25])
    ax_inset1.legend(loc = 2)
    axs[2].bar(range(1, len(B_mean)+1), B_mean)
    axs[2].set_xlabel('Fourier mode n')
    axs[2].set_ylabel('mean $B_n$ [K]')
    axs[2].set_xlim(0,72)
    axs[2].text(2,8,'(c)')
    ax_inset2 = axs[2].inset_axes([0.5, 0.67, 0.4, 0.3])
    ax_inset2.plot(r_keys[mask], Bn[mask, 1], label=r'$B_1$')
    ax_inset2.plot(r_keys[mask], Bn[mask, 2], label=r'$B_{2}$')
    ax_inset2.set_xlabel(r'$r$[\AA]')
    ax_inset2.set_ylabel(r'$B_n(r)$[K]')
    ax_inset2.legend(loc = 2)
    plt.savefig('../figures/DisorderVisualisation.pdf')